# Notebook 05 — SQL Server Integration

Run this notebook **locally on Windows** using Jupyter or VS Code.

It creates `ExamProctoringDB`, creates `ExamVideos` and `CheatingEvents`, registers videos, imports alert CSV files from Notebook 04B, and verifies the saved records.

## 1. Install packages

In [25]:
import cv2
import pandas as pd
import pyodbc

print("OpenCV:", cv2.__version__)
print("Pandas:", pd.__version__)
print("pyodbc:", pyodbc.version)

OpenCV: 5.0.0
Pandas: 2.3.3
pyodbc: 5.3.0


In [26]:
!pip install -q pyodbc pandas opencv-python

## 2. Imports and ODBC driver check

In [27]:
from pathlib import Path
import cv2
import pandas as pd
import pyodbc

print("pyodbc version:", pyodbc.version)
print("Available ODBC drivers:")
for d in pyodbc.drivers():
    print(" -", d)


pyodbc version: 5.3.0
Available ODBC drivers:
 - SQL Server
 - SQL Server Native Client RDA 11.0
 - ODBC Driver 18 for SQL Server
 - Microsoft Access Driver (*.mdb, *.accdb)
 - Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)
 - Microsoft Access Text Driver (*.txt, *.csv)
 - Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)


## 3. SQL Server settings

In [28]:
DRIVER = "ODBC Driver 18 for SQL Server"
SERVER = r"(localdb)\MSSQLLocalDB"   
DATABASE = "ExamProctoringDB"

TRUSTED_CONNECTION = True
SQL_USERNAME = ""
SQL_PASSWORD = ""

ENCRYPT = "yes"
TRUST_SERVER_CERTIFICATE = "yes"

print("Server:", SERVER)
print("Database:", DATABASE)


Server: (localdb)\MSSQLLocalDB
Database: ExamProctoringDB


## 4. Build connection string

In [29]:
def make_connection_string(database="master"):
    parts = [
        f"DRIVER={{{DRIVER}}}",
        f"SERVER={SERVER}",
        f"DATABASE={database}",
        f"Encrypt={ENCRYPT}",
        f"TrustServerCertificate={TRUST_SERVER_CERTIFICATE}",
    ]

    if TRUSTED_CONNECTION:
        parts.append("Trusted_Connection=yes")
    else:
        parts.append(f"UID={SQL_USERNAME}")
        parts.append(f"PWD={SQL_PASSWORD}")

    return ";".join(parts) + ";"


## 5. Test SQL Server connection

In [30]:
try:
    test_conn = pyodbc.connect(
        make_connection_string("master"),
        timeout=10
    )

    cur = test_conn.cursor()
    cur.execute("SELECT @@SERVERNAME, @@VERSION")
    row = cur.fetchone()

    print("Connected successfully.")
    print("Server name:", row[0])
    print(row[1])

    test_conn.close()

except Exception as e:
    print("Connection failed:")
    print(e)


Connected successfully.
Server name: DESKTOP-M70NM43\LOCALDB#BE639D11
Microsoft SQL Server 2019 (RTM-CU27-GDR) (KB5040948) - 15.0.4382.1 (X64) 
	Jul  1 2024 20:03:23 
	Copyright (C) 2019 Microsoft Corporation
	Express Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 19045: ) (Hypervisor)



## 6. Create database

In [31]:
master_conn = pyodbc.connect(
    make_connection_string("master"),
    autocommit=True
)

cur = master_conn.cursor()
cur.execute("SELECT DB_ID(?)", DATABASE)
exists = cur.fetchone()[0]

if exists is None:
    cur.execute(f"CREATE DATABASE [{DATABASE}]")
    print("Database created:", DATABASE)
else:
    print("Database already exists:", DATABASE)

master_conn.close()


Database already exists: ExamProctoringDB


## 7. Connect to project database

In [32]:
conn = pyodbc.connect(
    make_connection_string(DATABASE)
)

cursor = conn.cursor()

cursor.execute("SELECT DB_NAME()")
print("Current database:", cursor.fetchone()[0])


Current database: ExamProctoringDB


## 8. Create tables

In [33]:
schema_sql = """
IF OBJECT_ID('dbo.ExamVideos', 'U') IS NULL
BEGIN
    CREATE TABLE dbo.ExamVideos (
        VideoID INT IDENTITY(1,1) PRIMARY KEY,
        VideoName NVARCHAR(255) NOT NULL,
        VideoPath NVARCHAR(1000) NOT NULL,
        DurationSeconds FLOAT NULL,
        FPS FLOAT NULL,
        TotalFrames INT NULL,
        UploadDate DATETIME2 NOT NULL DEFAULT SYSDATETIME(),
        Status NVARCHAR(50) NOT NULL DEFAULT 'Processed'
    );
END;

IF OBJECT_ID('dbo.CheatingEvents', 'U') IS NULL
BEGIN
    CREATE TABLE dbo.CheatingEvents (
        EventID INT IDENTITY(1,1) PRIMARY KEY,
        VideoID INT NOT NULL,
        StudentTrackID INT NOT NULL,
        BehaviorDetected NVARCHAR(150) NOT NULL,
        ConfidenceScore FLOAT NULL,
        VideoTimeSeconds FLOAT NULL,
        EventDateTime DATETIME2 NULL,
        ScreenshotPath NVARCHAR(1000) NULL,
        HeadDirection NVARCHAR(50) NULL,
        SideLookDuration FLOAT NULL,
        HeadTurnCount INT NULL,
        PhoneHitsInWindow INT NULL,
        CONSTRAINT FK_CheatingEvents_ExamVideos
            FOREIGN KEY (VideoID)
            REFERENCES dbo.ExamVideos(VideoID)
    );
END;
"""

cursor.execute(schema_sql)
conn.commit()

print("Tables are ready.")


Tables are ready.


## 9. Verify tables

In [34]:
cursor.execute("""
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME
""")

tables = [r[0] for r in cursor.fetchall()]
tables


['CheatingEvents', 'ExamVideos']

## 10. Configure local project folder

In [35]:
from pathlib import Path

PROJECT_DIR = Path(
    r"D:\AI instant Training\cheating tracking"
)

VIDEO_DIR = PROJECT_DIR / "videos"
ROBUST_OUTPUT_DIR = PROJECT_DIR / "outputs" / "final_cheating_robust"
ALERT_DIR = ROBUST_OUTPUT_DIR / "alerts"
SCREENSHOT_DIR = ROBUST_OUTPUT_DIR / "screenshots"

print("PROJECT_DIR:", PROJECT_DIR)
print("VIDEO_DIR:", VIDEO_DIR)
print("ALERT_DIR:", ALERT_DIR)


PROJECT_DIR: D:\AI instant Training\cheating tracking
VIDEO_DIR: D:\AI instant Training\cheating tracking\videos
ALERT_DIR: D:\AI instant Training\cheating tracking\outputs\final_cheating_robust\alerts


## 11. Check local files

In [36]:
video_files = sorted(
    list(VIDEO_DIR.glob("*.mp4")) +
    list(VIDEO_DIR.glob("*.avi")) +
    list(VIDEO_DIR.glob("*.mov")) +
    list(VIDEO_DIR.glob("*.mkv"))
)

alert_csv_files = sorted(
    ALERT_DIR.glob("*_robust_alerts.csv")
)

print("Videos found:", len(video_files))
for v in video_files:
    print(" -", v.name)

print()
print("Alert CSV files found:", len(alert_csv_files))
for f in alert_csv_files:
    print(" -", f.name)


Videos found: 4
 - video1.mp4
 - video2.mp4
 - video3.mp4
 - video4.mp4

Alert CSV files found: 4
 - video1_robust_alerts.csv
 - video2_robust_alerts.csv
 - video3_robust_alerts.csv
 - video4_robust_alerts.csv


## 12. Video metadata helper

In [37]:
def get_video_metadata(video_path):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return {
            "duration": None,
            "fps": None,
            "frames": None
        }

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    duration = frames / fps if fps > 0 else 0

    cap.release()

    return {
        "duration": duration,
        "fps": fps,
        "frames": frames
    }


## 13. Insert videos

In [38]:
def get_or_create_video(video_path):
    video_path = Path(video_path).resolve()

    cursor.execute(
        """
        SELECT VideoID
        FROM dbo.ExamVideos
        WHERE VideoPath = ?
        """,
        str(video_path)
    )

    row = cursor.fetchone()

    if row:
        return int(row[0])

    meta = get_video_metadata(video_path)

    cursor.execute(
        """
        INSERT INTO dbo.ExamVideos
        (
            VideoName,
            VideoPath,
            DurationSeconds,
            FPS,
            TotalFrames,
            Status
        )
        OUTPUT INSERTED.VideoID
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        video_path.name,
        str(video_path),
        meta["duration"],
        meta["fps"],
        meta["frames"],
        "Processed"
    )

    video_id = int(cursor.fetchone()[0])
    conn.commit()

    return video_id


video_id_map = {}

for video_path in video_files:
    video_id = get_or_create_video(video_path)
    video_id_map[video_path.name] = video_id

    print(
        video_path.name,
        "-> VideoID:",
        video_id
    )


video1.mp4 -> VideoID: 1
video2.mp4 -> VideoID: 2
video3.mp4 -> VideoID: 3
video4.mp4 -> VideoID: 4


## 14. View ExamVideos

In [39]:
videos_df = pd.read_sql(
    """
    SELECT *
    FROM dbo.ExamVideos
    ORDER BY VideoID
    """,
    conn
)

videos_df


C:\Users\EL-Hussein Store\AppData\Local\Temp\ipykernel_27720\38620525.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  videos_df = pd.read_sql(


,VideoID,VideoName,VideoPath,DurationSeconds,FPS,TotalFrames,UploadDate,Status
0,1,video1.mp4,D:\AI instant Training\cheating tracking\video...,20.333333,24.000000,488,2026-10-28 12:58:31.211111,Processed
1,2,video2.mp4,D:\AI instant Training\cheating tracking\video...,10.885875,23.976024,261,2026-10-28 12:58:31.274111,Processed
2,3,video3.mp4,D:\AI instant Training\cheating tracking\video...,10.677344,23.976000,256,2026-10-28 12:58:31.316113,Processed
3,4,video4.mp4,D:\AI instant Training\cheating tracking\video...,9.426083,23.976024,226,2026-10-28 12:58:31.351119,Processed


In [40]:
print("Alert CSV files found:", len(alert_csv_files))

Alert CSV files found: 4


## 15. Duplicate check helper

In [41]:
def event_already_exists(
    video_id,
    student_track_id,
    behavior,
    video_time
):
    cursor.execute(
        """
        SELECT TOP 1 EventID
        FROM dbo.CheatingEvents
        WHERE
            VideoID = ?
            AND StudentTrackID = ?
            AND BehaviorDetected = ?
            AND ABS(VideoTimeSeconds - ?) < 0.001
        """,
        video_id,
        int(student_track_id),
        str(behavior),
        float(video_time)
    )

    return cursor.fetchone() is not None


## 16. Import alert CSV files

In [43]:
from pandas.errors import EmptyDataError

inserted_count = 0
skipped_count = 0

for csv_path in alert_csv_files:

    try:
        alerts_df = pd.read_csv(csv_path)

    except EmptyDataError:
        print(csv_path.name, "-> empty file, skipped")
        continue

    if alerts_df.empty:
        print(csv_path.name, "-> no alerts, skipped")
        continue

    print(csv_path.name, "->", len(alerts_df), "alerts")

    for _, row in alerts_df.iterrows():

        video_name = str(row["video"])

        if video_name not in video_id_map:
            print("Video not found locally:", video_name)
            continue

        video_id = video_id_map[video_name]

        if event_already_exists(
            video_id,
            row["student_track_id"],
            row["behavior_detected"],
            row["video_time_seconds"]
        ):
            skipped_count += 1
            continue

        event_datetime = None

        if pd.notna(row.get("event_datetime")):
            event_datetime = pd.to_datetime(
                row["event_datetime"]
            ).to_pydatetime()

        cursor.execute(
            """
            INSERT INTO dbo.CheatingEvents
            (
                VideoID,
                StudentTrackID,
                BehaviorDetected,
                ConfidenceScore,
                VideoTimeSeconds,
                EventDateTime,
                ScreenshotPath,
                HeadDirection,
                SideLookDuration,
                HeadTurnCount,
                PhoneHitsInWindow
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,

            int(video_id),
            int(row["student_track_id"]),
            str(row["behavior_detected"]),
            float(row["confidence_score"]),
            float(row["video_time_seconds"]),
            event_datetime,
            str(row.get("screenshot_path", "")),
            str(row.get("head_direction", "")),
            float(row.get("side_look_duration", 0) or 0),
            int(row.get("head_turn_count", 0) or 0),
            int(row.get("phone_hits_in_window", 0) or 0)
        )

        inserted_count += 1


conn.commit()

print()
print("Inserted events:", inserted_count)
print("Skipped duplicates:", skipped_count)

video1_robust_alerts.csv -> empty file, skipped
video2_robust_alerts.csv -> 3 alerts
video3_robust_alerts.csv -> 10 alerts
video4_robust_alerts.csv -> 2 alerts

Inserted events: 15
Skipped duplicates: 0


## 17. Full alert history

In [44]:
alert_history_df = pd.read_sql(
    """
    SELECT
        e.EventID,
        v.VideoName,
        e.StudentTrackID,
        e.BehaviorDetected,
        e.ConfidenceScore,
        e.VideoTimeSeconds,
        e.EventDateTime,
        e.ScreenshotPath,
        e.HeadDirection,
        e.SideLookDuration,
        e.HeadTurnCount,
        e.PhoneHitsInWindow
    FROM dbo.CheatingEvents e
    INNER JOIN dbo.ExamVideos v
        ON e.VideoID = v.VideoID
    ORDER BY e.EventID
    """,
    conn
)

alert_history_df


C:\Users\EL-Hussein Store\AppData\Local\Temp\ipykernel_27720\4283007450.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  alert_history_df = pd.read_sql(


,EventID,VideoName,StudentTrackID,BehaviorDetected,ConfidenceScore,VideoTimeSeconds,EventDateTime,ScreenshotPath,HeadDirection,SideLookDuration,HeadTurnCount,PhoneHitsInWindow
0,1,video2.mp4,3,Repeated head turns,0.7500,6.256,2026-10-28 08:33:02,/content/drive/MyDrive/cheating tracking/outpu...,left,0.125,3,0
1,2,video2.mp4,10,Prolonged sideways looking,0.7001,7.507,2026-10-28 08:33:03,/content/drive/MyDrive/cheating tracking/outpu...,left,2.502,0,0
2,3,video2.mp4,1,Prolonged sideways looking,0.7001,10.385,2026-10-28 08:33:05,/content/drive/MyDrive/cheating tracking/outpu...,left,2.503,0,0
3,4,video3.mp4,2,Phone detected,0.8000,0.667,2026-10-28 08:33:06,/content/drive/MyDrive/cheating tracking/outpu...,forward,0.000,0,8
4,5,video3.mp4,4,Phone detected,0.8000,0.667,2026-10-28 08:33:06,/content/drive/MyDrive/cheating tracking/outpu...,unknown,0.000,0,8
5,6,video3.mp4,2,Phone detected,0.8000,2.211,2026-10-28 08:33:07,/content/drive/MyDrive/cheating tracking/outpu...,left,0.083,0,8
6,7,video3.mp4,4,Phone detected,0.8000,2.211,2026-10-28 08:33:07,/content/drive/MyDrive/cheating tracking/outpu...,unknown,0.000,0,8
7,8,video3.mp4,3,Prolonged sideways looking,0.7001,3.754,2026-10-28 08:33:08,/content/drive/MyDrive/cheating tracking/outpu...,left,2.503,0,0
8,9,video3.mp4,8,Phone detected,0.8000,4.046,2026-10-28 08:33:08,/content/drive/MyDrive/cheating tracking/outpu...,unknown,0.000,0,11
9,10,video3.mp4,7,Phone detected,0.8000,4.129,2026-10-28 08:33:08,/content/drive/MyDrive/cheating tracking/outpu...,left,0.626,0,8


## 18. Alerts by behavior

In [45]:
behavior_summary = pd.read_sql(
    """
    SELECT
        BehaviorDetected,
        COUNT(*) AS NumberOfAlerts
    FROM dbo.CheatingEvents
    GROUP BY BehaviorDetected
    ORDER BY NumberOfAlerts DESC
    """,
    conn
)

behavior_summary


C:\Users\EL-Hussein Store\AppData\Local\Temp\ipykernel_27720\1939751015.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  behavior_summary = pd.read_sql(


,BehaviorDetected,NumberOfAlerts
0,Phone detected,9
1,Prolonged sideways looking,5
2,Repeated head turns,1


## 19. Alerts by video

In [46]:
video_summary = pd.read_sql(
    """
    SELECT
        v.VideoName,
        COUNT(e.EventID) AS NumberOfAlerts
    FROM dbo.ExamVideos v
    LEFT JOIN dbo.CheatingEvents e
        ON v.VideoID = e.VideoID
    GROUP BY v.VideoName
    ORDER BY v.VideoName
    """,
    conn
)

video_summary


C:\Users\EL-Hussein Store\AppData\Local\Temp\ipykernel_27720\1137301558.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  video_summary = pd.read_sql(


,VideoName,NumberOfAlerts
0,video1.mp4,0
1,video2.mp4,3
2,video3.mp4,10
3,video4.mp4,2


## 20. Inspect one student's history

In [47]:
QUERY_VIDEO = "video3.mp4"
QUERY_STUDENT_ID = 2

student_history = pd.read_sql(
    """
    SELECT
        e.EventID,
        v.VideoName,
        e.StudentTrackID,
        e.BehaviorDetected,
        e.ConfidenceScore,
        e.VideoTimeSeconds,
        e.EventDateTime,
        e.ScreenshotPath
    FROM dbo.CheatingEvents e
    INNER JOIN dbo.ExamVideos v
        ON v.VideoID = e.VideoID
    WHERE
        v.VideoName = ?
        AND e.StudentTrackID = ?
    ORDER BY e.VideoTimeSeconds
    """,
    conn,
    params=[
        QUERY_VIDEO,
        QUERY_STUDENT_ID
    ]
)

student_history


C:\Users\EL-Hussein Store\AppData\Local\Temp\ipykernel_27720\418068176.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  student_history = pd.read_sql(


,EventID,VideoName,StudentTrackID,BehaviorDetected,ConfidenceScore,VideoTimeSeconds,EventDateTime,ScreenshotPath
0,4,video3.mp4,2,Phone detected,0.8,0.667,2026-10-28 08:33:06,/content/drive/MyDrive/cheating tracking/outpu...
1,6,video3.mp4,2,Phone detected,0.8,2.211,2026-10-28 08:33:07,/content/drive/MyDrive/cheating tracking/outpu...
2,11,video3.mp4,2,Phone detected,0.8,5.339,2026-10-28 08:33:09,/content/drive/MyDrive/cheating tracking/outpu...
3,13,video3.mp4,2,Phone detected,0.8,6.381,2026-10-28 08:33:09,/content/drive/MyDrive/cheating tracking/outpu...


## 22. Close connection

In [48]:
conn.close()
print("SQL Server connection closed.")


SQL Server connection closed.
